In [ ]:
%load_ext autoreload
%autoreload 2

# Classical Pipeline to CNN/ViT Selector

## 1. Data preparation

Steps, in this order:

1. Load the dataset with kagglehub (version 4).
2. Remove exact duplicate images (MD5 hash).
3. Make binary labels: No DR vs. DR.
4. Split into train / val / test (70/15/15), stratified, seed 42.

In [ ]:
import time

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src import benchmark, data, models, preprocess, training

### 1.1 Load the dataset

In [ ]:
data_dir = data.download_dataset()
df_raw = data.load_metadata(data_dir)

print(f"Images loaded: {len(df_raw):,}")

### 1.2 Remove exact duplicates

- Groups whose copies have different 0-4 grades are dropped.
- Other groups keep one copy.

In [ ]:
df_unique, dup_stats = data.remove_exact_duplicates(df_raw)

for name, value in dup_stats.items():
    print(f"{name:<28} {value:>6,}")

### 1.3 Binary labels

Grade 0 = No DR. Grades 1-4 = DR.

In [ ]:
df_labeled = data.add_binary_labels(df_unique)

print("Images per grade (after duplicate removal):")
print(df_labeled.groupby(["grade", "label_name"]).size().rename("images").to_string())

counts = df_labeled["label_name"].value_counts()
print("\nImages per binary class:")
print(pd.DataFrame({"images": counts, "share": (counts / counts.sum()).round(3)}).to_string())

### 1.4 Stratified split

In [ ]:
df_split = data.stratified_split(df_labeled, label_col="label", seed=data.SEED)

table = pd.crosstab(df_split["split"], df_split["label_name"]).reindex(data.SPLITS)
table["total"] = table.sum(axis=1)
table["share of images"] = (table["total"] / table["total"].sum()).round(3)
table["DR share"] = (table["DR"] / table["total"]).round(3)

print("Split sizes and class counts:")
print(table.to_string())

### 1.5 Leakage check

No image should appear in more than one split, by ID or by file content (MD5).

In [ ]:
for col in ["id_code", "md5"]:
    splits_per_value = df_split.groupby(col)["split"].nunique()
    leaked = splits_per_value[splits_per_value > 1]
    print(f"{col}: {len(leaked)} values found in more than one split")
    assert leaked.empty, f"Leakage: {len(leaked)} {col} values are shared across splits"

print("\nNo overlap between splits.")

## 2. OpenCV preprocessing pipeline

**Order:** grayscale → CLAHE → Gaussian blur → Canny → closing

- **Grayscale first.** `cv2.imread` returns BGR, so the conversion uses `BGR2GRAY`.
- **CLAHE before blur.** It boosts local contrast so faint vessels stand out.
- **Blur before Canny.** It smooths the noise CLAHE amplifies, which would otherwise become false edges.
- **Closing last.** It bridges 1-pixel gaps in the edge lines.

**Canny thresholds: 40 / 120**, chosen by measuring gradients on 200 training images:

- Low 40 is just above the 75th percentile of gradient strength, so background texture is ignored.
- High 120 is between the 95th and 98th percentiles, so edges only start at vessels, the optic disc, lesions, and the rim.
- The 1:3 ratio is in Canny's recommended 1:2 to 1:3 range.

**Model input:** the CLAHE image, copied to 3 channels, 224x224, values 0 to 1.

In [ ]:
for stage, settings in preprocess.PIPELINE_CONFIG.items():
    print(f"{stage:<12} {settings}")

### 2.1 Every stage, for 3 training images

One image each from grades 0, 2, and 4.

In [ ]:
train_df = df_split[df_split["split"] == "train"]
examples = (
    train_df[train_df["grade"].isin([0, 2, 4])]
    .groupby("grade")
    .sample(1, random_state=data.SEED)
)

stage_titles = {
    "bgr": "Original",
    "gray": "Grayscale",
    "clahe": "CLAHE",
    "blurred": "Gaussian blur 5x5",
    "edges": "Canny 40/120",
    "closed": "Closing 3x3",
    "model_input": "Model input",
}

fig, axes = plt.subplots(
    len(examples), len(stage_titles), figsize=(2.6 * len(stage_titles), 2.8 * len(examples))
)
for row, (_, example) in enumerate(examples.iterrows()):
    stages = preprocess.run_pipeline(preprocess.load_bgr(example["path"]))
    for col, (key, title) in enumerate(stage_titles.items()):
        ax = axes[row, col]
        image = stages[key]
        if key == "bgr":
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))  # matplotlib expects RGB
        elif image.ndim == 2:
            ax.imshow(image, cmap="gray", vmin=0, vmax=255)  # fixed range: no auto-stretch
        else:
            ax.imshow(image)  # 3-channel float in [0, 1]
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(title, fontsize=10)
        if col == 0:
            ax.set_ylabel(f"grade {example['grade']} ({example['label_name']})", fontsize=10)

fig.tight_layout()
plt.show()

### 2.2 Normalization stats (training split only)

In [ ]:
mean, std = preprocess.compute_mean_std(train_df)

print(f"Training mean per channel: {mean.round(4)}")
print(f"Training std per channel:  {std.round(4)}")

### 2.3 PyTorch datasets

All three splits use the training mean and std.

In [ ]:
datasets = {
    name: preprocess.RetinaDataset(df_split[df_split["split"] == name], mean, std)
    for name in data.SPLITS
}
for name, dataset in datasets.items():
    print(f"{name:<5} {len(dataset):>5,} images")

image, label = datasets["train"][0]
print(f"\nOne item: shape {tuple(image.shape)}, {image.dtype}, label {label}")

loader = DataLoader(
    datasets["train"], batch_size=256, shuffle=True,
    generator=torch.Generator().manual_seed(data.SEED),
)
images, labels = next(iter(loader))
print(f"Training batch of {len(images)}: mean {images.mean():.3f}, std {images.std():.3f} (expect about 0 and 1)")

## 3. Models

- **Small CNN:** 3 conv blocks (16, 32, 64 filters, each with ReLU and max-pool) → global average pooling → fully connected 64 → 128 → 2. All weights train.
- **DeiT-tiny** (`deit_tiny_patch16_224`, pretrained): backbone frozen. Only the new 2-class head trains.

**Training settings:** CPU, batch size 32, `num_workers=0`, Adam (learning rate 0.001), cross-entropy loss.

**Model selection:** keep the epoch with the best **validation** accuracy. The test split is not used here.

In [ ]:
torch.manual_seed(data.SEED)
cnn = models.SmallCNN(num_classes=2)
deit = models.create_deit_tiny(num_classes=2)

param_counts = pd.DataFrame(
    [models.count_parameters(cnn), models.count_parameters(deit)],
    index=["Small CNN", "DeiT-tiny (frozen backbone)"],
)
print(param_counts.map("{:,}".format).to_string())

### 3.1 Small CNN: 5 epochs

In [ ]:
train_loader = training.make_loader(datasets["train"], shuffle=True, seed=data.SEED)
val_loader = training.make_loader(datasets["val"], shuffle=False)

cnn_history = training.fit(cnn, train_loader, val_loader, epochs=5)

### 3.2 DeiT-tiny: frozen features, then head only (20 epochs)

1. Run the frozen backbone once per image and keep its 192-value output.
2. Train only the head on those saved features.

In [ ]:
start = time.perf_counter()
features = {
    name: training.extract_features(deit, training.make_loader(datasets[name], shuffle=False))
    for name in ["train", "val"]  # test stays untouched until the final evaluation
}
print(f"Frozen features computed once, in {time.perf_counter() - start:.0f}s")
for name, feature_set in features.items():
    print(f"{name:<5} {tuple(feature_set.tensors[0].shape)}")

In [ ]:
head_history = training.fit(
    deit.head,
    training.make_loader(features["train"], shuffle=True, seed=data.SEED),
    training.make_loader(features["val"], shuffle=False),
    epochs=20,
)

In [ ]:
# Check: the head on cached features must match the full model on real images.
images, _ = next(iter(val_loader))  # val_loader isn't shuffled, so this is the first 32 val images
deit.eval()
with torch.no_grad():
    full_logits = deit(images)
    cached_logits = deit.head(features["val"].tensors[0][: len(images)])

max_diff = (full_logits - cached_logits).abs().max().item()
print(f"Largest logit difference, full model vs. cached features: {max_diff:.2e}")
assert torch.allclose(full_logits, cached_logits, atol=1e-4)

### 3.3 Best epochs

Chosen on validation accuracy. The test split is still unused.

In [ ]:
histories = {"Small CNN": cnn_history, "DeiT-tiny (head only)": head_history}

best_rows = {}
for name, history in histories.items():
    epochs_df = pd.DataFrame(history)
    best_rows[name] = epochs_df.loc[epochs_df["val_acc"].idxmax()]
best = pd.DataFrame(best_rows).T[["epoch", "train_acc", "val_acc", "val_loss"]]
best["epoch"] = best["epoch"].astype(int)

val_labels = df_split.loc[df_split["split"] == "val", "label"]
print("Best epoch per model (chosen on validation accuracy):")
print(best.to_string(float_format="{:.3f}".format))
print(f"\nMajority-class baseline on val: {val_labels.value_counts(normalize=True).max():.3f}")

## 4. Test-set comparison

The test split is used here for the first and only time.

- **Majority-class baseline:** always predicts the most common *training* label.
- **Sensitivity (DR recall):** the share of true DR images the model catches.
- **Size:** the saved weights (`state_dict`), in MB.
- **CPU latency:** one forward pass on a batch of 32 test images. Eval mode, no gradients, 3 warm-up runs, then the mean ± std of 10 timed runs.
  - DeiT-tiny is timed as the full model on images, not on cached features.
  - OpenCV preprocessing isn't included, only the model.

In [ ]:
test_loader = training.make_loader(datasets["test"], shuffle=False)
latency_batch, _ = next(iter(test_loader))
assert len(latency_batch) == training.BATCH_SIZE

print("Latency settings:")
print(f"  device   {benchmark.cpu_description()}")
print(f"  batch    {len(latency_batch)} images of shape {tuple(latency_batch.shape[1:])}")
print(f"  runs     {benchmark.WARMUP_RUNS} warm-up, then {benchmark.TIMED_RUNS} timed")

train_labels = df_split.loc[df_split["split"] == "train", "label"].to_numpy()
test_labels = df_split.loc[df_split["split"] == "test", "label"].to_numpy()

rows = {
    "Majority-class baseline": benchmark.majority_baseline(train_labels, test_labels),
    "Small CNN": benchmark.evaluate_model(cnn, test_loader, latency_batch),
    "DeiT-tiny (full model)": benchmark.evaluate_model(deit, test_loader, latency_batch),
}

In [ ]:
print("Comparison on the test split:")
print(benchmark.results_table(rows, batch_size=len(latency_batch)).to_string())